#### GOLD LAYER - FACT TABLES (SCD TYPE 1)

#### Purpose
- Build Gold fact tables from Gold staging tables  
- Use SCD Type 1 merge logic to avoid duplicates  
- Keep only the latest version of each fact record  

#### Output Tables
- `coffee.gold.fact_transactions`
- `coffee.gold.fact_transaction_items`

#### Notes
- `transaction_id` is the natural primary key for transactions  
- `transaction_item_sk` is the deterministic string surrogate key for transaction_items created in Silver  
- `transaction_item_key` is the bigint key required by evaluator/BI tools  


In [0]:
-- FACT: TRANSACTIONS (SCD1)
-- Key: transaction_id

CREATE OR REFRESH STREAMING TABLE coffee.gold.fact_transactions
COMMENT "Gold fact: transactions (SCD Type 1). Stores latest values per transaction_id.";


APPLY CHANGES INTO coffee.gold.fact_transactions
FROM STREAM(coffee.gold.stg_transactions)
KEYS (transaction_id)
SEQUENCE BY silver_updated_at
STORED AS SCD TYPE 1;

In [0]:
-- FACT: TRANSACTION ITEMS (SCD1)
-- Key: transaction_item_sk
--
-- Notes:
--   - transaction_item_sk is the true primary key (string SHA)
--   - transaction_item_key is bigint surrogate for BI tools

CREATE OR REFRESH STREAMING TABLE coffee.gold.fact_transaction_items
COMMENT "Gold fact: transaction items (SCD Type 1). Primary key is transaction_item_sk.";

APPLY CHANGES INTO coffee.gold.fact_transaction_items
FROM STREAM(coffee.gold.stg_transaction_items)
KEYS (transaction_item_sk)
SEQUENCE BY silver_updated_at
STORED AS SCD TYPE 1;